# Identical Twin Experiments & Performance Trade-offs

Welcome to the ninth interactive notebook of the `digital-twins` repository.

We introduce the **Identical Twin Experiment**. When developing a Data Assimilation (DA) algorithm, we rarely have access to the "True" state of the real world to verify if our algorithm is working. To solve this, we:
1. Run a highly detailed simulation (the "True System").
2. Extract artificial, noisy sensor readings from it.
3. Feed *only* those noisy readings to our Particle Filter.
4. Compare the Filter's estimated state against the "True System's" hidden state.

### The Ultimate Trade-off (Cost vs. Accuracy)
In discrete-event DT (like tracking the number of cars waiting at a One-Way Traffic Control light), we must run $N$ independent simulations during the *Predict* step of the Particle Filter.

If $N = 100$, the simulation runs blazingly fast, but our Root Mean Square Error (RMSE) will be terrible because we aren't covering the state space. 
If $N = 10,000$, our RMSE will be excellent, but the computer might take 5 minutes to calculate a 10-second real-world window—meaning **we failed the real-time requirement of DT**.

In this notebook, we simulate a one-way traffic queue. You will adjust the number of particles $N$ and instantly see the impact on Execution Time and RMSE.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, IntSlider

### Interactive Learning: Finding the Diminishing Returns

**The Scenario:** A construction crew has closed one lane of a two-lane road. A traffic light lets East-bound traffic go, then West-bound traffic. We have a noisy camera sensor counting cars departing the intersection, but we want to estimate the **hidden number of cars waiting in the West-bound queue**.

**Instructions:**
Adjust the `Num Particles (N)` slider. 
- Set it to **50**. Notice how jagged the blue estimation line is. The RMSE is high, but the execution time is basically 0.00 seconds.
- Set it to **5000**. Notice how perfectly the blue line tracks the black dashed line (Ground Truth). But look at the Execution Time—it has skyrocketed!
- **Your Goal:** Find the "sweet spot" (usually around $N=800$ to $1000$, as noted in Table 7.6) where the RMSE stops dropping significantly, so you aren't wasting CPU cycles.

In [ ]:
def run_identical_twin_experiment(N_particles=1000):
    # --- 1. Set Up the "True System" (Identical Twin) ---
    np.random.seed(42) # Fixed seed so the 'True World' is identical every time
    
    total_steps = 60
    true_west_queue = np.zeros(total_steps)
    noisy_measurements = np.zeros(total_steps)
    
    current_true_queue = 5.0
    
    # Generate Ground Truth & Noisy Sensor Data
    for k in range(total_steps):
        # Traffic arrives randomly (Poisson process approximated by Normal for simplicity)
        arrivals = max(0, np.random.normal(3.0, 1.5))
        
        # Traffic departs if the light is green (Light toggles every 10 steps)
        light_is_green = (k // 10) % 2 == 0
        departures = max(0, np.random.normal(5.0, 1.0)) if light_is_green else 0.0
        
        # Actual departures cannot exceed the queue
        actual_departures = min(current_true_queue + arrivals, departures)
        
        current_true_queue = current_true_queue + arrivals - actual_departures
        true_west_queue[k] = current_true_queue
        
        # The camera sensor counts departures but has heavy noise
        noisy_measurements[k] = max(0, actual_departures + np.random.normal(0, 2.0))
        
    # --- 2. Data Assimilation (Particle Filter) ---
    # We time this block to show the computational cost
    start_time = time.time()
    
    # Vectorized Particle Filter for performance
    # particles array holds the estimated queue size for N particles
    # We intentionally use a different seed here so the filter doesn't "cheat"
    np.random.seed(99) 
    particles = np.random.uniform(0, 10, N_particles)
    weights = np.ones(N_particles) / N_particles
    
    estimated_queue = np.zeros(total_steps)
    
    for k in range(total_steps):
        # A. PREDICT STEP (Using our imperfect Simulation Model)
        p_arrivals = np.maximum(0, np.random.normal(3.0, 2.0, N_particles)) # Model has slightly wrong noise
        light_is_green = (k // 10) % 2 == 0
        p_depart_capacity = np.maximum(0, np.random.normal(5.0, 1.5, N_particles)) if light_is_green else np.zeros(N_particles)
        
        p_actual_departures = np.minimum(particles + p_arrivals, p_depart_capacity)
        particles = particles + p_arrivals - p_actual_departures
        
        # Add Process Rejuvenation (Section 7.3.3) to prevent degeneracy
        particles += np.random.normal(0, 0.5, N_particles)
        particles = np.maximum(0, particles) # Queue can't be negative
        
        # B. UPDATE STEP (Weight based on Measurement Likelihood)
        # How well did the particle's predicted departure match the noisy camera measurement?
        variance = 2.0 ** 2
        diff = p_actual_departures - noisy_measurements[k]
        likelihoods = np.exp(-0.5 * (diff ** 2) / variance)
        
        weights *= likelihoods
        weight_sum = np.sum(weights)
        if weight_sum == 0 or np.isnan(weight_sum):
            weights = np.ones(N_particles) / N_particles
        else:
            weights /= weight_sum
            
        # Record the weighted average estimate
        estimated_queue[k] = np.average(particles, weights=weights)
        
        # C. RESAMPLE STEP (Systematic Resampling)
        cum_sum = np.cumsum(weights)
        cum_sum[-1] = 1.0
        indices = np.zeros(N_particles, dtype=int)
        r = np.random.uniform(0, 1.0 / N_particles)
        
        i = 0
        for j in range(N_particles):
            u = r + j / N_particles
            while u > cum_sum[i]:
                i += 1
            indices[j] = i
            
        particles = particles[indices]
        weights = np.ones(N_particles) / N_particles

    execution_time = time.time() - start_time
    
    # --- 3. Evaluate Performance (RMSE) ---
    rmse = np.sqrt(np.mean((true_west_queue - estimated_queue)**2))
    
    # --- 4. Plotting ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    
    # Top Plot: Queue Tracking
    time_arr = np.arange(total_steps)
    ax1.plot(time_arr, true_west_queue, 'k--', linewidth=2, label='True System (Hidden Ground Truth)')
    ax1.plot(time_arr, estimated_queue, 'b-o', linewidth=2, alpha=0.8, label=f'PF Estimated Queue (N={N_particles})')
    
    # Highlight Green/Red Light Phases
    for i in range(0, total_steps, 10):
        color = 'lightgreen' if (i // 10) % 2 == 0 else 'salmon'
        ax1.axvspan(i, min(i+10, total_steps-1), color=color, alpha=0.2)
        
    ax1.set_title("Identical Twin Experiment: West-Bound Traffic Queue Estimation", fontweight='bold')
    ax1.set_ylabel("Number of Vehicles in Queue")
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right')
    
    # Bottom Plot: Performance Metrics
    ax2.axis('off')
    ax2.text(0.1, 0.7, f"Filter Execution Time: {execution_time:.4f} seconds", fontsize=16, color='darkred' if execution_time > 0.05 else 'darkgreen')
    ax2.text(0.1, 0.3, f"Root Mean Square Error (RMSE): {rmse:.3f} vehicles", fontsize=16, color='darkred' if rmse > 1.5 else 'darkgreen')
    
    # Add context text for the student
    conclusion = "(Too few particles! High Error.)" if N_particles < 200 else "(Diminishing returns. Wasting CPU!)" if N_particles > 2000 else "(Sweet spot! Good balance of speed and accuracy.)"
    ax2.text(0.6, 0.5, conclusion, fontsize=14, fontstyle='italic', color='gray')
    
    plt.tight_layout()
    plt.show()

# Create the interactive widget
interact(run_identical_twin_experiment, 
         N_particles=IntSlider(value=1000, min=50, max=10000, step=50, description='Num Particles ($N$):', continuous_update=False, layout={'width':'600px'}));